## Addition Interp (2) !

This notebook is an attempt to explore how transformers learn to represent Fibonacci recurrence. In short, our question is : 

**"How do small transformers trained on addition data learn to add?"**

authored by : vorrjjard

##### Setup (Don't read, just run!)

In [13]:
import torch as t
from torch.utils.data import Dataset, DataLoader


import transformer_lens

from transformer_lens import HookedTransformer, ActivationCache, HookedTransformerConfig
from transformer_lens.train import HookedTransformerTrainConfig, train

import typing
from typing import Literal, List

from fib_interp_2.utils import generate_sample
from dataclasses import dataclass

import numpy as np

In [14]:
DEVICE = 'cuda' if t.cuda.is_available() else 'cpu'

##### 1. Data Generation

Related to Nanda's work on modular addition, we first start by creating a long tensor containing all our dataset examples.


In [15]:
@dataclass
class dataConfig:
    min_output: int = 0
    n_digits_input = 3
    n_digits_output = 4
    max_output: int = 999 
    n_samples: int = 1000
data_cfg = dataConfig()
print(data_cfg)

dataConfig(min_output=0, max_output=999, n_samples=1000)


In [16]:
vocab = {str(i): i for i in range(10)}
vocab['='] = 10

Let's define a small vocabulary for our project. Hence, our `d_vocab = 10` 

In [17]:
# 0 - 9, = : 10.

x = [vocab[char] for char in generate_sample(data_cfg)]

In [18]:
# Generate samples, tokenize, and stack on top of a new batch dimension.

samples_tensor = t.stack(
    [t.tensor([vocab[char] for char in generate_sample(data_cfg)]) for n in range(0, data_cfg.n_samples)]
    ,dim=0)

In [19]:
class SumDataset(Dataset):
    def __init__(self : Dataset, token_dict : dict, samples):
        self.token_dict = token_dict
        self.samples = samples

    def __getitem__(self, idx):
        return {"tokens": self.samples[idx, :]}
    
    def __len__(self):
        return self.samples.shape[0]

In [20]:
ds = SumDataset(vocab, samples_tensor)
dl = DataLoader(ds, batch_size=128, shuffle=True, drop_last=True)

##### 2. Model Configuration / Training

In [21]:
@dataclass
class Config(HookedTransformerConfig):
    d_model: int = 128
    debug: bool = True
    layer_norm_eps: float = 1e-5
    d_vocab: int = 11 # 0 through 9, plus '=' at index 10
    init_range: float = 0.02
    n_ctx: int = 11
    d_head: int = 32
    d_mlp: int = 512
    n_heads: int = 4
    n_layers: int = 1
    device: str = DEVICE

cfg = Config()
print(cfg)

Config(d_model=128, d_head=32, n_layers=1, n_ctx=11, n_heads=4, d_mlp=512, d_vocab=11, device='cpu', use_attn_result=False, use_split_qkv_input=False, default_prepend_bos=True, positional_embedding_type='standard', n_key_value_heads=None, attn_only=False, gated_mlp=False, uses_rms_norm=False, eps=1e-05, layer_norm_folding=False, act_fn='relu', normalization_type='LN', num_experts=None, experts_per_token=None, final_rms=False, dtype=torch.float32, model_name='custom', use_attn_scale=True, attn_scale=np.float64(5.656854249492381), use_hook_mlp_in=False, use_attn_in=False, use_qk_norm=False, use_local_attn=False, ungroup_grouped_query_attention=False, original_architecture=None, from_checkpoint=False, checkpoint_index=None, checkpoint_label_type=None, checkpoint_value=None, tokenizer_name=None, window_size=None, attn_types=None, init_mode='gpt2', n_devices=1, attention_dir='causal', seed=None, initializer_range=np.float64(0.07071067811865475), init_weights=True, scale_attn_by_inverse_laye

In [22]:
@dataclass 
class trainConfig(HookedTransformerTrainConfig):
    batch_size : int = 128
    device : str = DEVICE
    num_epochs : int = 1000
    lr : float = 0.001
    seed : int = 42
    save_every : int = 200
    save_dir : str = '../models/'
    wand : bool = False

train_cfg = trainConfig()
print(train_cfg)

trainConfig(num_epochs=1000, batch_size=128, lr=0.001, seed=42, momentum=0.0, max_grad_norm=None, weight_decay=None, optimizer_name='Adam', device='cpu', warmup_steps=0, save_every=200, save_dir='../models/', wandb=False, wandb_project_name=None, print_every=50, max_steps=None, wand=False)


In [23]:
model = HookedTransformer(cfg)

In [ ]:
model_cfg = 'model_0.pt'

In [ ]:
if not model_cfg:
    train(model, train_cfg, ds)
else:
    model = HookedTransformer(cfg)
    model.load_state_dict(t.load(model_cfg, map_location=cfg.device))

model.eval()

#### 3. Actual Interp

In [34]:
logits, cache = model.run_with_cache(samples_tensor[0, :].to(cfg.device))

ActivationCache with keys ['hook_embed', 'hook_pos_embed', 'blocks.0.hook_resid_pre', 'blocks.0.ln1.hook_scale', 'blocks.0.ln1.hook_normalized', 'blocks.0.attn.hook_q', 'blocks.0.attn.hook_k', 'blocks.0.attn.hook_v', 'blocks.0.attn.hook_attn_scores', 'blocks.0.attn.hook_pattern', 'blocks.0.attn.hook_z', 'blocks.0.hook_attn_out', 'blocks.0.hook_resid_mid', 'blocks.0.ln2.hook_scale', 'blocks.0.ln2.hook_normalized', 'blocks.0.mlp.hook_pre', 'blocks.0.mlp.hook_post', 'blocks.0.hook_mlp_out', 'blocks.0.hook_resid_post', 'ln_final.hook_scale', 'ln_final.hook_normalized', 'unembed.hook_in', 'unembed.hook_out']